# 🏦 Multi-Agent AI System
## Insurance Claims Processing & Fraud Detection

**Project Overview:**  
This notebook implements a multi-agent AI pipeline that simulates an intelligent insurance claims processing system.
Four specialised agents collaborate to:
1. Extract and validate claim details
2. Check policy eligibility
3. Detect potential fraud patterns
4. Generate a final recommendation report

**Framework:** AutoGen by Microsoft (v0.7+)  
**LLM Backend:** Google Gemini 2.5 Flash Lite (Free Tier)  
**Industry Context:** Insurance

## 🔧 Step 1: Install Libraries

In [ ]:
# -------------------------------------------------------
# autogen-agentchat is the modern package for AutoGen v0.7+
#       autogen-ext provides the model client for Gemini/OpenAI
# -------------------------------------------------------

# !pip install -q autogen-agentchat autogen-ext[openai] google-generativeai

## 📦 Step 2: Import Libraries

In [1]:
# -------------------------------------------------------
# WHAT: Import modern AutoGen classes
# WHY:  AutoGen v0.7+ uses autogen_agentchat
# OUTPUT: Confirms all imports loaded successfully
# -------------------------------------------------------

import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## 🔑 Step 3: Configure Gemini API

In [2]:
# -------------------------------------------------------
# WHAT: Configure the Gemini 2.5 Flash Lite model client
# WHY:  AutoGen v0.7+ uses a model client object instead of llm_config dict
#       Gemini exposes an OpenAI-compatible endpoint so we use OpenAIChatCompletionClient
# OUTPUT: Confirms the Gemini model client is ready to use
# -------------------------------------------------------

GEMINI_API_KEY = "API KEY"  # <-- API key

model_client = OpenAIChatCompletionClient(
    model="gemini-2.5-flash-lite",
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    model_info={
        "vision": True,
        "function_calling": True,
        "json_output": True,
        "family": "unknown",
    }
)

print("✅ Gemini model client ready!")

✅ Gemini model client ready!


C:\Users\Default.DESKTOP-73FCCDO\Documents\Installations\envs\metlife_agents\Lib\site-packages\autogen_ext\models\openai\_openai_client.py:466: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(self._model_info)


## 📋 Step 4: Define the Insurance Claim

This claim contains a few suspicious patterns.

In [3]:
# -------------------------------------------------------
# WHAT: Define a sample insurance claim (Scenario 1 - suspicious)
# WHY:  This is the input that flows through all four agents
# OUTPUT: Prints the claim for review before processing
# -------------------------------------------------------

claim = """
INSURANCE CLAIM DETAILS:
- Claim ID        : CLM-2024-00892
- Policy Number   : POL-MTL-44821
- Policyholder    : James Carter
- Policy Type     : Health Insurance
- Policy Start    : 2022-03-15
- Claim Date      : 2024-11-02
- Claim Amount    : $18,500
- Reason          : Emergency hospitalisation — chest pain, cardiac monitoring
- Hospital        : St. Mary's General Hospital, New York
- Treating Doctor : Dr. Priya Nair (Cardiologist)
- Prior Claims    : 3 claims in last 12 months totalling $4,200
- Notes           : Claim submitted 45 days after discharge.
                    Policyholder upgraded coverage 2 months before this claim.
"""

print("📄 Claim 1 loaded (Suspicious Scenario):")
print(claim)

📄 Claim 1 loaded (Suspicious Scenario):

INSURANCE CLAIM DETAILS:
- Claim ID        : CLM-2024-00892
- Policy Number   : POL-MTL-44821
- Policyholder    : James Carter
- Policy Type     : Health Insurance
- Policy Start    : 2022-03-15
- Claim Date      : 2024-11-02
- Claim Amount    : $18,500
- Reason          : Emergency hospitalisation — chest pain, cardiac monitoring
- Hospital        : St. Mary's General Hospital, New York
- Treating Doctor : Dr. Priya Nair (Cardiologist)
- Prior Claims    : 3 claims in last 12 months totalling $4,200
- Notes           : Claim submitted 45 days after discharge.
                    Policyholder upgraded coverage 2 months before this claim.



## 🤖 Step 5: Create the Four Specialist Agents

| Agent | Role |
|---|---|
| Claims_Extractor | Reads and structures the raw claim |
| Policy_Validator | Checks coverage and eligibility |
| Fraud_Detector | Scores fraud risk, lists red flags |
| Senior_Reviewer | Synthesises all findings → final decision |

In [4]:
# -------------------------------------------------------
# WHAT: Define four specialised AssistantAgents
# WHY:  Dividing tasks between agents improves accuracy
#       Each agent is an expert in one area, like a real claims department
# OUTPUT: Four agent objects ready to work as a team
# -------------------------------------------------------

claims_extractor = AssistantAgent(
    name="Claims_Extractor",
    model_client=model_client,
    system_message="""
    You are a Claims Extraction Specialist at XYZ Insurance.
    Read the raw insurance claim and extract key details into a clean structured summary:
    Claim ID, Policyholder, Claim Amount, Reason, Date, Hospital, Doctor, and any notable observations.
    Be factual and concise. Use bullet points.
    """
)

policy_validator = AssistantAgent(
    name="Policy_Validator",
    model_client=model_client,
    system_message="""
    You are a Policy Validation Officer at XYZ Insurance.
    Assess the claim for policy eligibility:
    - Is the policy active at the time of the claim?
    - Does the treatment fall under standard health insurance coverage?
    - Are the claim timelines reasonable?
    State your result as ELIGIBLE, REQUIRES REVIEW, or INELIGIBLE with clear reasons.
    """
)

fraud_detector = AssistantAgent(
    name="Fraud_Detector",
    model_client=model_client,
    system_message="""
    You are a Fraud Detection Analyst at XYZ Insurance.
    Identify red flags in the claim. Key fraud indicators:
    recent policy upgrades before large claims, delayed submissions,
    high claim frequency, unusually large amounts, documentation inconsistencies.
    Give a FRAUD RISK SCORE from 1-10 (1=very low, 10=very high) and list all red flags found.
    """
)

senior_reviewer = AssistantAgent(
    name="Senior_Reviewer",
    model_client=model_client,
    system_message="""
    You are the Senior Claims Review Manager at XYZ Insurance.
    Synthesise findings from all agents and produce a FINAL CLAIMS REPORT:
    1. Brief claim summary
    2. Policy eligibility status
    3. Fraud risk level
    4. Final Recommendation: APPROVE / APPROVE WITH CONDITIONS / INVESTIGATE / REJECT
    5. Clear reasoning
    End your response with exactly the word: TERMINATE
    """
)

print("✅ Four agents created successfully!")
print("   → Claims_Extractor")
print("   → Policy_Validator")
print("   → Fraud_Detector")
print("   → Senior_Reviewer")

✅ Four agents created successfully!
   → Claims_Extractor
   → Policy_Validator
   → Fraud_Detector
   → Senior_Reviewer


## 🚀 Step 6: Run Pipeline - Claim 1 (Suspicious)

Agents process the claim in sequence using **RoundRobinGroupChat** -  each agent reads the full conversation and adds its analysis.

In [5]:
# -------------------------------------------------------
# WHAT: Assemble the agent team and run the claims pipeline
# WHY:  RoundRobinGroupChat sends the conversation to each agent in turn
#       TextMentionTermination stops the loop when Senior_Reviewer says TERMINATE
# OUTPUT: Full agent conversation with final claims report
# -------------------------------------------------------

termination = TextMentionTermination("TERMINATE")

claims_team = RoundRobinGroupChat(
    participants=[claims_extractor, policy_validator, fraud_detector, senior_reviewer],
    termination_condition=termination
)

task = f"""
Process the following MetLife insurance claim through the full review pipeline.
Each agent should analyse it based on their specialisation.

{claim}
"""

print("🔍 Processing Claim 1 — Suspicious Scenario")
print("=" * 60)

await Console(claims_team.run_stream(task=task))

🔍 Processing Claim 1 — Suspicious Scenario
---------- TextMessage (user) ----------

Process the following MetLife insurance claim through the full review pipeline.
Each agent should analyse it based on their specialisation.


INSURANCE CLAIM DETAILS:
- Claim ID        : CLM-2024-00892
- Policy Number   : POL-MTL-44821
- Policyholder    : James Carter
- Policy Type     : Health Insurance
- Policy Start    : 2022-03-15
- Claim Date      : 2024-11-02
- Claim Amount    : $18,500
- Reason          : Emergency hospitalisation — chest pain, cardiac monitoring
- Hospital        : St. Mary's General Hospital, New York
- Treating Doctor : Dr. Priya Nair (Cardiologist)
- Prior Claims    : 3 claims in last 12 months totalling $4,200
- Notes           : Claim submitted 45 days after discharge.
                    Policyholder upgraded coverage 2 months before this claim.


---------- TextMessage (Claims_Extractor) ----------
Here is a structured summary of the MetLife insurance claim:

*   **Claim

TaskResult(messages=[TextMessage(id='c48da362-ed1f-4eea-b482-832931d77241', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 5, 12, 23, 51, 59, 631269, tzinfo=datetime.timezone.utc), content="\nProcess the following MetLife insurance claim through the full review pipeline.\nEach agent should analyse it based on their specialisation.\n\n\nINSURANCE CLAIM DETAILS:\n- Claim ID        : CLM-2024-00892\n- Policy Number   : POL-MTL-44821\n- Policyholder    : James Carter\n- Policy Type     : Health Insurance\n- Policy Start    : 2022-03-15\n- Claim Date      : 2024-11-02\n- Claim Amount    : $18,500\n- Reason          : Emergency hospitalisation — chest pain, cardiac monitoring\n- Hospital        : St. Mary's General Hospital, New York\n- Treating Doctor : Dr. Priya Nair (Cardiologist)\n- Prior Claims    : 3 claims in last 12 months totalling $4,200\n- Notes           : Claim submitted 45 days after discharge.\n                    Policyholder upgraded covera

- **Claims Extractor** → Successfully structured the raw claim into key fields for downstream agents.
- **Policy Validator** → Flagged as **REQUIRES REVIEW**: policy is active and treatment is covered, but 45-day submission delay needs verification against policy terms.
- **Fraud Detector** → **Fraud Risk Score: 7/10 (HIGH)**: identified 4 red flags:
  - Recent policy upgrade just 2 months before a large claim.
  - Delayed submission of 45 days for an emergency hospitalisation.
  - Unusually large claim amount of $18,500.
  - Pattern of 3 prior claims in 12 months alongside other suspicious factors.
- **Senior Reviewer** → Final Decision: **INVESTIGATE**: claim is not rejected outright as the medical emergency is legitimate, but the combination of red flags requires deeper verification before approval.

**Key Takeaway:** The multi-agent system intelligently identified a high-risk claim and recommended investigation rather than blindly approving or rejecting it.

## ✅ Step 7: Run Pipeline - Claim 2 (Low Risk)

A routine, clean claim for comparison.  

In [6]:
# -------------------------------------------------------
# WHAT: Process a second clean, low-risk claim
# WHY:  Demonstrates the system handles both normal and suspicious claims differently
# OUTPUT: Lower fraud score, likely APPROVE recommendation
# -------------------------------------------------------

claim_2 = """
INSURANCE CLAIM DETAILS:
- Claim ID        : CLM-2024-00901
- Policy Number   : POL-MTL-33201
- Policyholder    : Sarah Obi
- Policy Type     : Health Insurance
- Policy Start    : 2020-07-01
- Claim Date      : 2024-10-15
- Claim Amount    : $850
- Reason          : Annual health check and blood tests
- Hospital        : City Medical Clinic, Chicago
- Treating Doctor : Dr. Alan Smith (General Practitioner)
- Prior Claims    : 1 claim in last 12 months totalling $400
- Notes           : Claim submitted 5 days after appointment.
                    Long-standing policy, no unusual activity.
"""

claims_team_2 = RoundRobinGroupChat(
    participants=[claims_extractor, policy_validator, fraud_detector, senior_reviewer],
    termination_condition=TextMentionTermination("TERMINATE")
)

task_2 = f"""
Process the following MetLife insurance claim through the full review pipeline.
Each agent should analyse it based on their specialisation.

{claim_2}
"""

print("🔍 Processing Claim 2 — Low Risk Scenario")
print("=" * 60)

await Console(claims_team_2.run_stream(task=task_2))

🔍 Processing Claim 2 — Low Risk Scenario
---------- TextMessage (user) ----------

Process the following MetLife insurance claim through the full review pipeline.
Each agent should analyse it based on their specialisation.


INSURANCE CLAIM DETAILS:
- Claim ID        : CLM-2024-00901
- Policy Number   : POL-MTL-33201
- Policyholder    : Sarah Obi
- Policy Type     : Health Insurance
- Policy Start    : 2020-07-01
- Claim Date      : 2024-10-15
- Claim Amount    : $850
- Reason          : Annual health check and blood tests
- Hospital        : City Medical Clinic, Chicago
- Treating Doctor : Dr. Alan Smith (General Practitioner)
- Prior Claims    : 1 claim in last 12 months totalling $400
- Notes           : Claim submitted 5 days after appointment.
                    Long-standing policy, no unusual activity.


---------- TextMessage (Claims_Extractor) ----------
Here is a structured summary of the MetLife insurance claim:

*   **Claim ID**: CLM-2024-00901
*   **Policyholder**: Sarah 

TaskResult(messages=[TextMessage(id='8533a409-a0e0-493d-bbf9-a1adb09df6d1', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 5, 12, 23, 52, 28, 808586, tzinfo=datetime.timezone.utc), content='\nProcess the following MetLife insurance claim through the full review pipeline.\nEach agent should analyse it based on their specialisation.\n\n\nINSURANCE CLAIM DETAILS:\n- Claim ID        : CLM-2024-00901\n- Policy Number   : POL-MTL-33201\n- Policyholder    : Sarah Obi\n- Policy Type     : Health Insurance\n- Policy Start    : 2020-07-01\n- Claim Date      : 2024-10-15\n- Claim Amount    : $850\n- Reason          : Annual health check and blood tests\n- Hospital        : City Medical Clinic, Chicago\n- Treating Doctor : Dr. Alan Smith (General Practitioner)\n- Prior Claims    : 1 claim in last 12 months totalling $400\n- Notes           : Claim submitted 5 days after appointment.\n                    Long-standing policy, no unusual activity.\n\n', type='TextM

- **Claims Extractor** → Cleanly summarised a straightforward routine claim with no anomalies.
- **Policy Validator** → Flagged as **REQUIRES REVIEW**: only to confirm preventive care coverage details in the policy schedule, no red flags.
- **Fraud Detector** → **Fraud Risk Score: 1/10 (LOW)**: zero red flags identified:
  - Modest claim amount of $850 consistent with routine services.
  - Prompt submission within 5 days.
  - Long-standing policy since 2020 with clean history.
  - Only 1 prior claim in 12 months.
- **Senior Reviewer** → Final Decision: **APPROVE**: routine preventive care, clean policy history, low fraud risk, prompt submission.

**Key Takeaway:** The same pipeline correctly identified this as a clean, low-risk claim and recommended approval, demonstrating the system's ability to differentiate between genuine and suspicious claims intelligently.